# Bronze – Raw Screenshot Preprocessing

End-to-end visualization of the Bronze pipeline:
1. **Load** a raw game screenshot
2. **Denoise** (Non-Local Means)
3. **Enhance contrast** (CLAHE on L-channel)
4. **Sharpen** (unsharp mask)
5. **Extract metadata** (brightness, contrast, etc.)

Uses `src.pipeline.bronze` and `src.config`.

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.config import BRONZE_DIR, SILVER_DIR
from src.pipeline.bronze import (
    H, W,
    denoise,
    enhance_contrast,
    extract_metadata,
    load_image,
    preprocess,
    process_image,
    sharpen,
)

sns.set_theme(style="darkgrid")
%matplotlib inline

In [ ]:
SCREENSHOT_DIR = BRONZE_DIR
OUTPUT_DIR = SILVER_DIR

pngs = sorted(SCREENSHOT_DIR.glob("*.png"))
if not pngs:
    raise FileNotFoundError(
        f"No PNG screenshots found in {SCREENSHOT_DIR}. "
        "Place game screenshots in data/bronze/ and restart."
    )

print(f"Found {len(pngs)} screenshot(s)")
print(f"Using: {pngs[0].name}")
image_path = str(pngs[0])

## 1.  Load & Validate

In [ ]:
raw = load_image(image_path)
print(f"Shape  : {raw.shape}")
print(f"Dtype  : {raw.dtype}")
print(f"Range  : {raw.min()} – {raw.max()}")

## 2.  Pipeline Step-by-Step

In [ ]:
stages = {
    "Original": raw,
    "1. Denoise": denoise(raw),
    "2. CLAHE": enhance_contrast(denoise(raw)),
    "3. Sharpen (final)": preprocess(raw),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, (title, img) in zip(axes.ravel(), stages.items()):
    ax.imshow(img)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.axis("off")
fig.suptitle("Bronze Preprocessing Pipeline", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 3.  Per-pixel Difference Map

Shows *where* the pipeline changed the image most.

In [ ]:
cleaned = preprocess(raw)
diff = np.abs(raw.astype(np.int16) - cleaned.astype(np.int16)).astype(np.uint8)
diff_gray = diff.max(axis=2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(raw)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(diff_gray, cmap="hot", vmin=0, vmax=255)
axes[1].set_title("Per-pixel Change Magnitude")
axes[1].axis("off")

axes[2].imshow(cleaned)
axes[2].set_title("Preprocessed")
axes[2].axis("off")

print(f"Mean per-pixel change : {diff.mean():.2f}")
print(f"Max  per-pixel change : {diff.max()}")
plt.tight_layout()
plt.show()

## 4.  Histogram Comparison (RGB channels)

In [ ]:
colors = ("Red", "Green", "Blue")
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for i, (img, title) in enumerate([(raw, "Original"), (cleaned, "Preprocessed")]):
    ax = axes[i]
    for c, color_name in enumerate(colors):
        hist = cv2.calcHist([img], [c], None, [256], [0, 256])
        ax.plot(hist, color=color_name.lower(), label=color_name, alpha=0.8)
    ax.set_title(title)
    ax.set_xlim(0, 256)
    if i == 0:
        ax.set_ylabel("Pixel count")
    ax.legend()

fig.suptitle("RGB Histogram — Before vs After", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5.  Metadata

In [ ]:
meta = extract_metadata(cleaned, image_path=str(Path(image_path).resolve()))
print(meta)

## 6.  End-to-End Processing

Runs `process_image` and saves artifacts to `data/silver/`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
result = process_image(image_path, str(OUTPUT_DIR))

print(f"Preprocessed PNG : {result['preprocessed']}")
print(f"Metadata JSON    : {result['metadata']}")

with open(result["metadata"]) as f:
    saved_meta = json.load(f)
print(f"\nMetadata keys: {list(saved_meta.keys())}")

## 7.  Load Saved Artifacts (Verification)

In [ ]:
reloaded = load_image(result["preprocessed"])
np.testing.assert_array_equal(cleaned, reloaded)
print("Round-trip assertion passed: saved == reloaded")